In [ ]:
import os
import json
import pandas as pd

import google_auth_oauthlib.flow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

### List out items on a youtube playlist, given
- client_secret.json that details the oauth credentials
- playlist id

In [ ]:
# Disable OAuthlib's HTTPS verification when running locally.
# *DO NOT* leave this option enabled in production.
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

api_service_name = "youtube"
api_version = "v3"
client_secrets_file = "client_secret.json"
scopes = ["https://www.googleapis.com/auth/youtube.readonly"]
token_file = "token.json"


def main():
    credentials = None

    # checks for existing credentials from previous run
    if os.path.exists(token_file):
        credentials = Credentials.from_authorized_user_file(token_file, scopes)

    # if no valid credentials are available, get new credentials and create an API client
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else: # else, get credentials and create an API client
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                client_secrets_file, scopes)
            credentials = flow.run_local_server(port=0)

        # save credentials for next run
        with open(token_file, "w") as token:
            token.write(credentials.to_json())

    youtube = googleapiclient.discovery.build(
        api_service_name, api_version, credentials=credentials)

    try: 
        # test with 11/25/28 playlist
        request = youtube.playlistItems().list(
            part="snippet,contentDetails",
            maxResults=5,
            playlistId="PL2vR6i7V0rIoXdJ7HZT67afj4T24Ji0Od"
        )
        response = request.execute()

        # extract song/item name from the nested maps
        # 'items' key is list of maps -> each map has 'snippet' key -> has 'title' key, which is the song name as the value
        items = response.get("items", [])

        song_data = []

        # iterate through the items maps to extract each song
        for item in items:
            snippet = item.get("snippet", {})
            title = snippet.get("title", "")
            song_data.append(title)

        # create a DataFrame from the song data
        df = pd.DataFrame(song_data, columns=["Song_Name"])

        print(df)
    except HttpError as e:
        print(f"An HTTP error {e.resp.status} occurred:\n{e.content}")

    # print(json.dumps(response, indent=4))

In [22]:
if __name__ == "__main__":
    main()

{
    "kind": "youtube#playlistItemListResponse",
    "etag": "VgPupGE9r_HMGwCSqodmT2JjBFo",
    "nextPageToken": "EAAaHlBUOkNBVWlFREV5UlVaQ00wSXhRelUzUkVVMFJURQ",
    "items": [
        {
            "kind": "youtube#playlistItem",
            "etag": "wJ9g-VfP35qR0qcpGrsMSUc2Xl8",
            "id": "UEwydlI2aTdWMHJJb1hkSjdIWlQ2N2FmajRUMjRKaTBPZC41NkI0NEY2RDEwNTU3Q0M2",
            "snippet": {
                "publishedAt": "2018-11-25T08:28:29Z",
                "channelId": "UC2-TrYd0f4tenPd45Cbmtqg",
                "title": "Foster The People - Sit Next to Me (Audio)",
                "description": "Official Audio for \u201cSit Next to Me\u201d by Foster The People\nListen to Foster The People: https://FosterThePeople.lnk.to/listenYD\n\nSubscribe to the official Foster The People YouTube channel: https://FosterThePeople.lnk.to/subscribeYD\n\nWatch more of Foster The People's music videos: https://FosterThePeople.lnk.to/listenYD/youtube\n\nFollow Foster The People:\nFacebook: htt